# 02 — Feature Engineering

## From EDA to Engineering Decisions

The following decisions are directly informed by findings in `01_eda.ipynb`:

| EDA Finding | Engineering / Modeling Decision |
|---|---|
| 10 legitimate sessions per user | Unsupervised one-class modeling — too few samples for supervised classification |
| Legitimate and impostor distributions overlap at population level | 88 per-user models, not one global model — separation only exists within individual users |
| No single feature separates classes even per-user | Multivariate feature vector — model evaluates joint distribution of all signals simultaneously |
| `false_enters` not discriminative between classes | Included as session noise feature, not relied upon as signal |
| Every session contains a malformed metadata record in mouse events | All parsers filter records missing the `Event` field before any computation |
| High variance in mouse event counts (mean 403, std 342) | Robust aggregates used alongside means — extreme values capped before velocity computation |

## Model Selection

Three unsupervised one-class models will be trained and compared:

- **Isolation Forest** — primary model. Trains on legitimate sessions only, produces continuous anomaly score, handles multivariate feature spaces well
- **One-Class SVM** — secondary model. Better suited for users with tight, well-defined legitimate distributions
- **Local Outlier Factor** — tertiary model. Density-based, no training phase, different enough to be a meaningful comparison

All models are trained on legitimate sessions only. Ground truth labels are held out and used exclusively for evaluation via False Rejection Rate, False Acceptance Rate, and Equal Error Rate per user.

## Feature Engineering Goals

Extract one fixed-length feature vector per session from raw key and mouse events:

- **Keystroke features** — dwell time (mean, std, min, max), flight time (mean, std, min, max), key event count
- **Mouse features** — velocity (mean, std), trajectory length, click dwell, directness ratio, hover time
- **Session features** — total session duration, field transition time, backspace rate, false_enters

In [1]:
import json
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

RAW_DATA_PATH = '../data/raw_kmt_dataset'

print("Libraries loaded.")
print(f"Users available: {len([f for f in os.listdir(RAW_DATA_PATH) if f.endswith('.json')])}")

Libraries loaded.
Users available: 88


In [ ]:
#Keystroke feature extractor
def extract_keystroke_features(key_events):
    """
    Extract dwell time and flight time features from raw key events.
    
    Dwell time  = duration between key pressed and key released (same key)
    Flight time = gap between key released and next key pressed (transition speed)
    """
    # Filter clean events only
    key_events = [e for e in key_events if 'Event' in e and 'Epoch' in e]
    
    pressed = {}
    dwell_times = []
    release_epochs = []
    press_epochs = []
    backspace_count = 0

    for event in key_events:
        key = event['Key']
        epoch = float(event['Epoch'])
        action = event['Event']

        if action == 'pressed':
            press_epochs.append(epoch)
            if key in ('backspace', 'Backspace'):
                backspace_count += 1
            pressed[key] = epoch

        elif action == 'released' and key in pressed:
            dwell = epoch - pressed[key]
            if 0 <= dwell <= 2.0:  # cap at 2 seconds
                dwell_times.append(dwell)
            release_epochs.append(epoch)
            del pressed[key]

    # Flight times from consecutive release -> press pairs
    release_epochs_sorted = sorted(release_epochs)
    press_epochs_sorted = sorted(press_epochs)

    flight_times = []
    for i in range(min(len(release_epochs_sorted), len(press_epochs_sorted)) - 1):
        flight = press_epochs_sorted[i+1] - release_epochs_sorted[i]
        if 0 < flight < 2.0:
            flight_times.append(flight)

    # Session duration from key events
    all_epochs = [float(e['Epoch']) for e in key_events]
    duration = max(all_epochs) - min(all_epochs) if len(all_epochs) > 1 else 0

    return {
        # Dwell time features
        'dwell_mean': np.mean(dwell_times) if dwell_times else 0,
        'dwell_std': np.std(dwell_times) if dwell_times else 0,
        'dwell_min': np.min(dwell_times) if dwell_times else 0,
        'dwell_max': np.max(dwell_times) if dwell_times else 0,
        'dwell_median': np.median(dwell_times) if dwell_times else 0,
        # Flight time features
        'flight_mean': np.mean(flight_times) if flight_times else 0,
        'flight_std': np.std(flight_times) if flight_times else 0,
        'flight_min': np.min(flight_times) if flight_times else 0,
        'flight_max': np.max(flight_times) if flight_times else 0,
        'flight_median': np.median(flight_times) if flight_times else 0,
        # Session level
        'key_event_count': len(key_events),
        'backspace_rate': backspace_count / len(key_events) if key_events else 0,
        'key_duration': duration
    }

# Test on one session
with open(os.path.join(RAW_DATA_PATH, 'raw_kmt_user_0001.json'), 'r') as f:
    test_user = json.load(f)

test_session = test_user['true_data']['test_1']
features = extract_keystroke_features(test_session['key_events'])
print("Keystroke features for user 0001 test_1:")
for k, v in features.items():
    print(f"  {k:20s}: {v:.6f}")

Keystroke features for user 0001 test_1:
  dwell_mean          : 0.093341
  dwell_std           : 0.036234
  dwell_min           : 0.036046
  dwell_max           : 0.288763
  dwell_median        : 0.087173
  flight_mean         : 0.330486
  flight_std          : 0.498293
  flight_min          : 0.024577
  flight_max          : 1.874097
  flight_median       : 0.094892
  key_event_count     : 118.000000
  backspace_rate      : 0.059322
  key_duration        : 17.075644


In [3]:
# mouse feature extractor
def extract_mouse_features(mouse_events):
    """
    Extract velocity, trajectory, click dwell, and hover features
    from raw mouse events.
    """
    # Filter clean events only — drop malformed metadata records
    mouse_events = [e for e in mouse_events if 'Event' in e and 'Epoch' in e]

    movement_events = [e for e in mouse_events if e['Event'] == 'movement']
    click_press_events = [e for e in mouse_events if 'press' in e['Event']]
    click_release_events = [e for e in mouse_events if 'release' in e['Event']]

    # --- Velocity and trajectory ---
    velocities = []
    trajectory_length = 0.0

    for i in range(1, len(movement_events)):
        prev = movement_events[i-1]
        curr = movement_events[i]

        dx = curr['Coordinates'][0] - prev['Coordinates'][0]
        dy = curr['Coordinates'][1] - prev['Coordinates'][1]
        dt = float(curr['Epoch']) - float(prev['Epoch'])

        distance = np.sqrt(dx**2 + dy**2)
        trajectory_length += distance

        if dt > 0:
            velocities.append(distance / dt)

    # --- Directness ratio ---
    # Straight line distance from first to last movement point
    # divided by total path length
    directness = 0.0
    if len(movement_events) >= 2 and trajectory_length > 0:
        first = movement_events[0]['Coordinates']
        last = movement_events[-1]['Coordinates']
        straight_line = np.sqrt(
            (last[0] - first[0])**2 + (last[1] - first[1])**2
        )
        directness = straight_line / trajectory_length

    # --- Click dwell ---
    # Match press and release events by button type
    click_dwells = []
    press_dict = {}
    for e in mouse_events:
        event_type = e['Event']
        epoch = float(e['Epoch'])
        if 'press' in event_type:
            press_dict[event_type.split()[0]] = epoch  # e.g. 'left'
        elif 'release' in event_type:
            button = event_type.split()[0]
            if button in press_dict:
                dwell = epoch - press_dict[button]
                if 0 <= dwell <= 2.0:
                    click_dwells.append(dwell)
                del press_dict[button]

    # --- Session duration from mouse events ---
    all_epochs = [float(e['Epoch']) for e in mouse_events]
    duration = max(all_epochs) - min(all_epochs) if len(all_epochs) > 1 else 0

    return {
        # Velocity features
        'mouse_velocity_mean': np.mean(velocities) if velocities else 0,
        'mouse_velocity_std': np.std(velocities) if velocities else 0,
        'mouse_velocity_max': np.max(velocities) if velocities else 0,
        'mouse_velocity_median': np.median(velocities) if velocities else 0,
        # Trajectory features
        'trajectory_length': trajectory_length,
        'directness_ratio': directness,
        # Click features
        'click_dwell_mean': np.mean(click_dwells) if click_dwells else 0,
        'click_dwell_std': np.std(click_dwells) if click_dwells else 0,
        'click_count': len(click_press_events),
        # Session level
        'mouse_event_count': len(mouse_events),
        'mouse_duration': duration
    }

# Test on one session
mouse_features = extract_mouse_features(test_session['mouse_events'])
print("Mouse features for user 0001 test_1:")
for k, v in mouse_features.items():
    print(f"  {k:25s}: {v:.6f}")

Mouse features for user 0001 test_1:
  mouse_velocity_mean      : 599.142377
  mouse_velocity_std       : 978.625739
  mouse_velocity_max       : 7108.231746
  mouse_velocity_median    : 249.894041
  trajectory_length        : 744.345138
  directness_ratio         : 0.821871
  click_dwell_mean         : 0.062383
  click_dwell_std          : 0.010238
  click_count              : 4.000000
  mouse_event_count        : 106.000000
  mouse_duration           : 20.291349


In [4]:
# Extractor and false_enter function
def extract_session_features(session, user_id, session_id, label):
    """
    Combine keystroke, mouse, and session-level features
    into a single fixed-length feature vector.
    """
    # Extract false_enters before filtering mouse events
    false_enters = 0
    for e in session['mouse_events']:
        if 'false_enters' in e:
            false_enters = e['false_enters']

    keystroke_features = extract_keystroke_features(session['key_events'])
    mouse_features = extract_mouse_features(session['mouse_events'])

    return {
        'user_id': user_id,
        'session_id': session_id,
        'label': label,
        **keystroke_features,
        **mouse_features,
        'false_enters': false_enters
    }

# Test on one session
combined = extract_session_features(
    test_session, 
    user_id='0001', 
    session_id='test_1', 
    label='legitimate'
)

print(f"Total features per session: {len(combined) - 3}")  # subtract user_id, session_id, label
print(f"\nFeature vector:")
for k, v in combined.items():
    if k not in ('user_id', 'session_id', 'label'):
        print(f"  {k:25s}: {v:.6f}")

Total features per session: 25

Feature vector:
  dwell_mean               : 0.093341
  dwell_std                : 0.036234
  dwell_min                : 0.036046
  dwell_max                : 0.288763
  dwell_median             : 0.087173
  flight_mean              : 0.330486
  flight_std               : 0.498293
  flight_min               : 0.024577
  flight_max               : 1.874097
  flight_median            : 0.094892
  key_event_count          : 118.000000
  backspace_rate           : 0.059322
  key_duration             : 17.075644
  mouse_velocity_mean      : 599.142377
  mouse_velocity_std       : 978.625739
  mouse_velocity_max       : 7108.231746
  mouse_velocity_median    : 249.894041
  trajectory_length        : 744.345138
  directness_ratio         : 0.821871
  click_dwell_mean         : 0.062383
  click_dwell_std          : 0.010238
  click_count              : 4.000000
  mouse_event_count        : 106.000000
  mouse_duration           : 20.291349
  false_enters         

In [5]:
# Build full feature dataset across all 88 users
all_features = []

for filename in tqdm(sorted(os.listdir(RAW_DATA_PATH)), desc="Processing users"):
    if not filename.endswith('.json'):
        continue

    user_id = filename.replace('raw_kmt_user_', '').replace('.json', '')

    with open(os.path.join(RAW_DATA_PATH, filename), 'r') as f:
        data = json.load(f)

    for label, sessions in [('legitimate', data['true_data']), ('impostor', data['false_data'])]:
        for session_id, session in sessions.items():
            features = extract_session_features(session, user_id, session_id, label)
            all_features.append(features)

features_df = pd.DataFrame(all_features)

print(f"Feature matrix shape: {features_df.shape}")
print(f"\nNull values per feature:")
print(features_df.isnull().sum()[features_df.isnull().sum() > 0])
print("\nSample:")
print(features_df.head(3).to_string())

Processing users: 100%|██████████| 88/88 [00:07<00:00, 11.07it/s]

Feature matrix shape: (1760, 28)

Null values per feature:
Series([], dtype: int64)

Sample:
  user_id session_id       label  dwell_mean  dwell_std  dwell_min  dwell_max  dwell_median  flight_mean  flight_std  flight_min  flight_max  flight_median  key_event_count  backspace_rate  key_duration  mouse_velocity_mean  mouse_velocity_std  mouse_velocity_max  mouse_velocity_median  trajectory_length  directness_ratio  click_dwell_mean  click_dwell_std  click_count  mouse_event_count  mouse_duration  false_enters
0    0001     test_1  legitimate    0.093341   0.036234   0.036046   0.288763      0.087173     0.330486    0.498293    0.024577    1.874097       0.094892              118        0.059322     17.075644           599.142377          978.625739         7108.231746             249.894041         744.345138          0.821871          0.062383         0.010238            4                106       20.291349             0
1    0001     test_2  legitimate    0.085055   0.023216   0.02402

In [6]:
# Save feature matrix
OUTPUT_PATH = '../data/features_kmt.csv'
features_df.to_csv(OUTPUT_PATH, index=False)
print(f"Feature matrix saved to {OUTPUT_PATH}")
print(f"Shape: {features_df.shape}")

Feature matrix saved to ../data/features_kmt.csv
Shape: (1760, 28)


In [7]:
# Feature distribution summary
feature_cols = [c for c in features_df.columns if c not in ('user_id', 'session_id', 'label')]

summary = features_df[feature_cols].describe().T
summary['skew'] = features_df[feature_cols].skew()
summary['zeros_pct'] = (features_df[feature_cols] == 0).mean() * 100

print("Feature health check:")
print(summary[['mean', 'std', 'min', 'max', 'skew', 'zeros_pct']].round(3).to_string())

Feature health check:
                           mean        std       min          max    skew  zeros_pct
dwell_mean                0.116      0.026     0.062        0.215   0.415      0.000
dwell_std                 0.044      0.029     0.009        0.296   2.206      0.000
dwell_min                 0.062      0.024     0.000        0.128  -0.362      0.795
dwell_max                 0.297      0.172     0.088        1.740   2.160      0.000
dwell_median              0.106      0.026     0.058        0.222   0.616      0.000
flight_mean               0.429      0.194     0.000        1.543   0.581      2.955
flight_std                0.417      0.143     0.000        0.752  -1.366      3.239
flight_min                0.046      0.080     0.000        1.543  10.662      2.955
flight_max                1.581      0.500     0.000        2.000  -1.896      2.955
flight_median             0.265      0.198     0.000        1.543   1.696      2.955
key_event_count          98.047     31.815 

## Feature Health Findings

| Issue | Features Affected | Fix |
|---|---|---|
| Extreme outliers | `mouse_velocity_std`, `mouse_velocity_max`, `mouse_duration` | Log transform + cap at 99th percentile |
| High skew (>5) | `key_event_count`, `trajectory_length`, `click_count`, `mouse_event_count` | Log transform |
| Zero inflation (>50%) | `backspace_rate`, `false_enters` | Keep as-is — zeros are meaningful signal |
| Low skew, well behaved | `dwell_*`, `flight_mean`, `mouse_velocity_median` | Scale only |

**Scaling strategy:** RobustScaler (median/IQR based) applied per-user after log transformation.  
StandardScaler would be corrupted by the extreme velocity outliers.

In [8]:
# log transformation to the severely skewed features 
from sklearn.preprocessing import RobustScaler
import warnings
warnings.filterwarnings('ignore')

# Features requiring log transform due to high skew or extreme outliers
log_transform_features = [
    'mouse_velocity_std',
    'mouse_velocity_max',
    'mouse_duration',
    'key_event_count',
    'trajectory_length',
    'click_count',
    'mouse_event_count',
    'key_duration',
    'flight_min'
]

# Cap at 99th percentile before log transform to handle extreme outliers
feature_cols = [c for c in features_df.columns if c not in ('user_id', 'session_id', 'label')]

transformed_df = features_df.copy()

for col in log_transform_features:
    cap = transformed_df[col].quantile(0.99)
    transformed_df[col] = transformed_df[col].clip(upper=cap)
    transformed_df[col] = np.log1p(transformed_df[col])  # log1p handles zeros safely

# Check skew after transformation
print("Skew before vs after log transform:")
print(f"{'Feature':<25} {'Before':>10} {'After':>10}")
print("-" * 47)
for col in log_transform_features:
    before = features_df[col].skew()
    after = transformed_df[col].skew()
    print(f"{col:<25} {before:>10.3f} {after:>10.3f}")

Skew before vs after log transform:
Feature                       Before      After
-----------------------------------------------
mouse_velocity_std            27.067      0.510
mouse_velocity_max            31.979      0.505
mouse_duration                20.419      0.725
key_event_count                6.540      1.440
trajectory_length              5.141      1.004
click_count                    8.899      0.320
mouse_event_count              5.339      0.347
key_duration                   4.730      0.628
flight_min                    10.662      1.773


In [9]:
# Apply RobustScaler per user
# Important: fit scaler on legitimate sessions only, transform all sessions
# This mirrors real deployment — you only know what normal looks like

from sklearn.preprocessing import RobustScaler

scaled_records = []

for user_id, user_data in transformed_df.groupby('user_id'):
    
    legitimate = user_data[user_data['label'] == 'legitimate']
    all_sessions = user_data.copy()
    
    scaler = RobustScaler()
    
    # Fit on legitimate only
    scaler.fit(legitimate[feature_cols])
    
    # Transform all sessions (legitimate + impostor)
    all_sessions[feature_cols] = scaler.transform(all_sessions[feature_cols])
    
    scaled_records.append(all_sessions)

scaled_df = pd.concat(scaled_records).reset_index(drop=True)

print(f"Scaled feature matrix shape: {scaled_df.shape}")
print(f"\nSample — user 0001 legitimate sessions:")
print(scaled_df[
    (scaled_df['user_id'] == '0001') & 
    (scaled_df['label'] == 'legitimate')
][feature_cols].round(3).to_string())

Scaled feature matrix shape: (1760, 28)

Sample — user 0001 legitimate sessions:
   dwell_mean  dwell_std  dwell_min  dwell_max  dwell_median  flight_mean  flight_std  flight_min  flight_max  flight_median  key_event_count  backspace_rate  key_duration  mouse_velocity_mean  mouse_velocity_std  mouse_velocity_max  mouse_velocity_median  trajectory_length  directness_ratio  click_dwell_mean  click_dwell_std  click_count  mouse_event_count  mouse_duration  false_enters
0       0.444      0.301     -0.051      0.430        -0.010        0.084       0.140      15.315       0.545          0.181            2.559           2.341         2.205               -0.749               0.205               0.668                 -2.991             -0.146            39.595            -1.723            0.310        1.000              0.361           2.219           0.0
1      -1.419     -0.795     -0.702     -0.654         0.089       -0.624       0.556      -0.868       0.252         -1.593            1.2

In [10]:
# Save scaled feature matrix
SCALED_PATH = '../data/features_kmt_scaled.csv'
scaled_df.to_csv(SCALED_PATH, index=False)
print(f"Scaled feature matrix saved to {SCALED_PATH}")
print(f"Shape: {scaled_df.shape}")
print(f"\nFiles in data/:")
for f in os.listdir('../data'):
    print(f"  {f}")

Scaled feature matrix saved to ../data/features_kmt_scaled.csv
Shape: (1760, 28)

Files in data/:
  .gitkeep
  features_kmt.csv
  features_kmt_scaled.csv
  feature_kmt_dataset
  raw_kmt_dataset


## Summary

Two feature matrices produced and saved to `data/`:

| File | Description |
|---|---|
| `features_kmt.csv` | Raw extracted features — 1,760 sessions × 25 features |
| `features_kmt_scaled.csv` | Log-transformed + RobustScaler applied per user — ready for modeling |

### Scaling decisions
- Log transform + 99th percentile cap applied to 9 high-skew features
- RobustScaler fit on legitimate sessions only per user — mirrors real deployment
- Scaler never sees impostor sessions during fitting

### Features carried forward into modeling (25 total)
- **Keystroke (13):** dwell mean/std/min/max/median, flight mean/std/min/max/median, key event count, backspace rate, key duration
- **Mouse (11):** velocity mean/std/max/median, trajectory length, directness ratio, click dwell mean/std, click count, mouse event count, mouse duration  
- **Session (1):** false_enters

